# Triangle rasterisation with a z-buffer

*graphics · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/graphics/triangle_rasterisation)

The way games draw 3D: project a triangle mesh onto the screen and compute the barycentric weights of every (pixel, triangle) pair at once. A min-reduction over triangles is the z-buffer. The same weights then drive flat, Gouraud and Phong shading, and texture lookup. Interpolating texture coordinates with $\lambda/z$ (perspective-correct) instead of $\lambda$ (affine) is what keeps a checkered floor straight.

**Vector form**

$$\mathbf q = \sum_k \lambda_k \mathbf v_k,\ \ \sum_k \lambda_k = 1 \qquad \frac{1}{z} = \sum_k \frac{\lambda_k}{z_k} \qquad a = \frac{\sum_k \lambda_k a_k / z_k}{\sum_k \lambda_k / z_k} \qquad I = k_a + k_d\,\mathbf n\cdot\mathbf l + k_s\,(\mathbf n\cdot\mathbf h)^s$$

**Index (tensor) form**

$$\lambda_{ptk} = \frac{\varepsilon_{ij}\,(v_{t,k+1} - q_p)_i\,(v_{t,k+2} - q_p)_j}{2A_t} \qquad \frac{1}{z_{pt}} = \frac{\lambda_{ptk}}{z_{tk}} \qquad t^*_p = \arg\min_t z_{pt} \qquad a_{pc} = \frac{\lambda_{pk}\,a_{pkc} / z_{pk}}{\lambda_{pk} / z_{pk}}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_{ni}$ | `scene['verts']` | (N, 3) | mesh vertices plus the 4 floor corners; faces $(T, 3)$ index into them |
| $R_{ij}\,x_{nj}$ | `einsum('ij,nj->ni', R, verts)` | (N, 3) | repeated $j$ ⇒ summed: spin the mesh (one rotation per frame) |
| $v_{tki},\ z_{tk}$ | `project(verts, cam)[faces]` | (T, 3, 2), (T, 3) | screen position and depth of corner $k$ of triangle $t$ (pinhole camera $f\,(x, y)/z$) |
| $A_t$ | `signed_area(screen)` | (T,) | half of $\varepsilon_{ij}(v_1 - v_0)_i (v_2 - v_0)_j$; $A_t \le 0$ ⇒ back face, culled |
| $\lambda_{ptk}$ | `barycentric(q, tri)` | (P, T, 3) | barycentric weights of every pixel in every triangle; computed in tiles of pixels to bound memory |
| $\lambda_{ptk}/z_{tk}$ | `einsum('ptk,tk->pt', lam, 1/z)` | (P, T) | repeated $k$ ⇒ summed: $1/z$ is what varies linearly on the screen |
| $t^*_p$ | `depth.min(dim=1)` | (P,) | the z-buffer: nearest covering triangle per pixel (pixels outside are $+\infty$) |
| $a_{pc}$ | `perspective_interp(lam, z, a)` | (P, C) | texture coordinates, normals, positions, Gouraud brightness interpolated with weights $\lambda/z$ |
| $\lambda_{pk}\,a_{pkc}$ | `einsum('pk,pkc->pc', lam, uv)` | (P, 2) | affine interpolation: right for flat-on triangles, wrong for tilted ones |

Barycentric weights are signed areas: $\lambda_k$ is the area of the small triangle opposite corner $k$ divided by the whole, so the three add up to 1 and all are positive exactly inside. Here they are computed for every pixel and every triangle at once, the brute-force version of what a GPU does in hardware. Tiling the pixels keeps the $(P, T, 3)$ tensor under a few million entries; a real rasteriser only visits each triangle's bounding box. The sign of the screen-space area also gives back-face culling for free: about half of a closed mesh faces away. Depth along a triangle is not linear on the screen, but $1/z$ is, so both the z-buffer test and every interpolated attribute use weights $\lambda_k/z_k$. The floor is only two triangles, so affine interpolation (PlayStation-1 style) visibly bends its checker lines, while the perspective-correct version keeps them straight. Flat shading shows every facet. Gouraud lights the corners and blends the brightness, which smears out the specular highlight. Phong blends the normals and lights every pixel, which keeps the highlight sharp.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/graphics/triangle_rasterisation/main.py
"""Main loop: parameters → mesh, camera and pixels → rasterise every frame → plot."""

import time
from pathlib import Path

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    res = 40 if fast_mode() else params["resolution"]
    n_frames = 5 if fast_mode() else params["frames"]
    scene = tensors.scene(params["mesh"], dev)
    cam = tensors.camera(res, dev)
    q = tensors.pixels(res, dev)  # (P, 2)
    print(f"{params['mesh']}: {len(scene['verts'])} vertices, {len(scene['faces'])} triangles (2 of them the floor)")
    print(
        f"{res}×{res} = {len(q)} pixels → barycentrics for up to {len(q) * len(scene['faces']):,} pixel-triangle pairs"
    )

    start = time.perf_counter()
    frames, stats = solve.animate_mesh(scene, cam, q, tensors.light(dev), n_frames, params["tilt"])
    elapsed = time.perf_counter() - start
    culled = sum(s["culled"] for s in stats) / len(stats)
    covered = sum(s["covered"] for s in stats) / len(stats)
    print(f"rendered {n_frames} frames in {elapsed:.1f} s ({1000 * elapsed / n_frames:.0f} ms per frame)")
    print(f"back-face culling removes {culled:.0f} of {stats[0]['triangles']} triangles per frame on average")
    print(f"surface covers {100 * covered / len(q):.0f}% of the pixels")
    print(f"largest perspective-vs-affine texture-coordinate gap on the floor: {stats[0]['uv_error']:.2f}")
    print("(uv runs 0 → 1 across the floor, so the affine checker is off by up to that many floor widths)")
    return plot.animate(frames, params["mesh"], out_dir / "triangle_rasterisation.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/graphics/triangle_rasterisation/equation.py
r"""Rasterisation → tensor translation.

A triangle t with screen corners v_{tki} (k = corner 0, 1, 2; i = screen x, y) covers pixel q_{pi} when all three
barycentric weights are non-negative. With the 2D Levi-Civita symbol ε_{ij} (ε_{01} = 1, ε_{10} = −1),

    λ_{ptk} = ε_{ij} (v_{t,k+1} − q_p)_i (v_{t,k+2} − q_p)_j / (2 A_t),
    2 A_t = ε_{ij} (v_{t1} − v_{t0})_i (v_{t2} − v_{t0})_j

(i, j summed): each λ is the area of the small triangle opposite corner k over the whole area, so λ_{pt0} + λ_{pt1}
+ λ_{pt2} = 1. A_t > 0 means the corners appear counter-clockwise, i.e. the triangle faces the camera; A_t ≤ 0 is a
back face and is culled.

Depth is not linear on the screen, but 1/z is: 1/z_{pt} = λ_{ptk} / z_{tk} (k summed). The z-buffer keeps, for every
pixel, the triangle with the smallest z — a min-reduction over t. Any attribute a (texture coordinate, normal,
colour) is interpolated perspective-correctly by weighting with λ/z,

    a_p = (λ_{pk} a_{pk} / z_{pk}) / (λ_{pk} / z_{pk}),

while the cheap affine version a_p = λ_{pk} a_{pk} (used by early game consoles) bends textures on big, tilted
triangles.
"""

import math

import torch

EPS2 = torch.tensor([[0.0, 1.0], [-1.0, 0.0]])  # ε_{ij}, 2D Levi-Civita symbol


def rotation(theta: float, tilt: float, device: torch.device) -> torch.Tensor:
    """R_{ij} = (spin by θ about y) · (tilt about x): (3, 3)."""
    c, s, ct, st = math.cos(theta), math.sin(theta), math.cos(tilt), math.sin(tilt)
    spin = torch.tensor([[c, 0, s], [0, 1, 0], [-s, 0, c]])
    tilt_m = torch.tensor([[1, 0, 0], [0, ct, -st], [0, st, ct]])
    return (spin @ tilt_m).to(device)


def rotate(R: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    r"""x'_{ni} = R_{ij} x_{nj}   (j summed).   (N, 3) → (N, 3)"""
    return torch.einsum("ij,nj->ni", R, x)


def project(x: torch.Tensor, cam: dict) -> tuple[torch.Tensor, torch.Tensor]:
    r"""Camera coordinates c_{ni} = R_{ij}(x_{nj} − e_j), then the pinhole: screen = c₀ + f (c_x, c_y) / c_z.

    Returns screen positions (N, 2) and depths z_n = c_z (N,)."""
    c = torch.einsum("ij,nj->ni", cam["R"], x - cam["eye"])
    z = c[:, 2]
    return cam["c"] + cam["f"] * c[:, :2] / z[:, None], z


def cross2(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    r"""ε_{ij} a_i b_j   (i, j summed) for any leading shape: (..., 2), (..., 2) → (...)"""
    return torch.einsum("ij,...i,...j->...", EPS2.to(a), a, b)


def signed_area(tri: torch.Tensor) -> torch.Tensor:
    r"""A_t = ½ ε_{ij} (v_{t1} − v_{t0})_i (v_{t2} − v_{t0})_j: positive when the corners are counter-clockwise.

    tri (T, 3, 2) → (T,)"""
    return 0.5 * cross2(tri[:, 1] - tri[:, 0], tri[:, 2] - tri[:, 0])


def barycentric(q: torch.Tensor, tri: torch.Tensor) -> torch.Tensor:
    r"""λ_{ptk} = ε_{ij}(v_{t,k+1} − q_p)_i (v_{t,k+2} − q_p)_j / 2A_t for every pixel and triangle at once.

    q (P, 2), tri (T, 3, 2) → (P, T, 3). The three edge terms add up to 2A_t, which normalises them."""
    d = tri[None] - q[:, None, None]  # (P, T, 3, 2): corners relative to the pixel
    lam = cross2(d.roll(-1, dims=2), d.roll(-2, dims=2))  # (P, T, 3): twice the sub-triangle areas
    return lam / lam.sum(-1, keepdim=True)


def inverse_depth(lam: torch.Tensor, z: torch.Tensor) -> torch.Tensor:
    r"""1/z_{pt} = λ_{ptk} / z_{tk}   (k summed): 1/z, not z, is linear on the screen.   (P, T, 3), (T, 3) → (P, T)"""
    return torch.einsum("ptk,tk->pt", lam, 1 / z)


def perspective_interp(lam: torch.Tensor, z: torch.Tensor, a: torch.Tensor) -> torch.Tensor:
    r"""a_{pc} = (λ_{pk} a_{pkc} / z_{pk}) / (λ_{pk} / z_{pk})   (k summed).   (P, 3), (P, 3), (P, 3, C) → (P, C)"""
    w = lam / z
    return torch.einsum("pk,pkc->pc", w, a) / w.sum(-1, keepdim=True)


def affine_interp(lam: torch.Tensor, a: torch.Tensor) -> torch.Tensor:
    r"""a_{pc} = λ_{pk} a_{pkc}   (k summed): linear in screen space, wrong under perspective.   → (P, C)"""
    return torch.einsum("pk,pkc->pc", lam, a)


def blinn_phong(n: torch.Tensor, to_light: torch.Tensor, to_eye: torch.Tensor) -> torch.Tensor:
    r"""I = k_a + k_d max(0, n_i l_i) + k_s max(0, n_i h_i)^s with the half vector h = (l + v)/|l + v|.

    n, to_eye (..., 3) unit vectors, to_light (3,) → (...)"""
    h = to_light + to_eye
    h = h / h.norm(dim=-1, keepdim=True)
    diffuse = (n * to_light).sum(-1).clamp(min=0)
    specular = (n * h).sum(-1).clamp(min=0) ** 40
    return 0.15 + 0.7 * diffuse + 0.5 * specular * (diffuse > 0)


def checker(uv: torch.Tensor, tiles: int = 8) -> torch.Tensor:
    r"""Checkerboard texture: 1 where ⌊n u⌋ + ⌊n v⌋ is even, else 0.   (P, 2) → (P,)"""
    cells = torch.floor(tiles * uv).sum(-1)
    return (cells % 2 == 0).float()


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/graphics/triangle_rasterisation/tensors.py
"""Build the concrete tensors: a procedural mesh (torus, icosphere or terrain) on a checkered floor, the camera and the
pixel grid.

Every triangle carries its corner texture coordinates uv_{tkc} (k = corner, c = u or v), so seams need no duplicated
vertices. Faces are wound counter-clockwise seen from outside (right-hand rule gives the outward normal).
"""

import math

import torch


def grid_faces(nu: int, nv: int, wrap_u: bool, wrap_v: bool) -> tuple[torch.Tensor, torch.Tensor]:
    """Two triangles per cell of an nu × nv parameter grid: faces (T, 3) and corner uv (T, 3, 2), T = 2·nu·nv."""
    cols_v = nv if wrap_v else nv + 1  # vertices per grid row

    def vid(i, j):
        return (i % nu if wrap_u else i) * cols_v + (j % nv if wrap_v else j)

    faces, uv = [], []
    for i in range(nu):
        for j in range(nv):
            a, b, c, d = (i, j), (i, j + 1), (i + 1, j + 1), (i + 1, j)  # this order makes ∂v × ∂u the normal
            for tri in ((a, b, c), (a, c, d)):
                faces.append([vid(*p) for p in tri])
                uv.append([[p[0] / nu, p[1] / nv] for p in tri])
    return torch.tensor(faces), torch.tensor(uv)


def torus(nu: int = 20, nv: int = 10, big_r: float = 0.9, small_r: float = 0.38):
    """Torus around the y axis: vertices (nu·nv, 3), faces (T, 3), corner uv (T, 3, 2)."""
    u = torch.arange(nu) * 2 * math.pi / nu
    v = torch.arange(nv) * 2 * math.pi / nv
    uu, vv = torch.meshgrid(u, v, indexing="ij")
    ring = big_r + small_r * torch.cos(vv)
    verts = torch.stack([ring * torch.cos(uu), small_r * torch.sin(vv), ring * torch.sin(uu)], -1).reshape(-1, 3)
    faces, uv = grid_faces(nu, nv, True, True)
    return verts, faces, uv * torch.tensor([3.0, 1.0])  # the long way round gets 3× more checker tiles


def icosphere(subdivisions: int = 2, radius: float = 1.0):
    """Icosahedron split `subdivisions` times (each triangle → 4) and pushed onto a sphere."""
    t = (1 + math.sqrt(5)) / 2
    verts = torch.tensor(
        [[-1, t, 0], [1, t, 0], [-1, -t, 0], [1, -t, 0], [0, -1, t], [0, 1, t], [0, -1, -t], [0, 1, -t]]
        + [[t, 0, -1], [t, 0, 1], [-t, 0, -1], [-t, 0, 1]],
        dtype=torch.float32,
    )
    faces = torch.tensor(
        [[0, 11, 5], [0, 5, 1], [0, 1, 7], [0, 7, 10], [0, 10, 11], [1, 5, 9], [5, 11, 4], [11, 10, 2], [10, 7, 6]]
        + [[7, 1, 8], [3, 9, 4], [3, 4, 2], [3, 2, 6], [3, 6, 8], [3, 8, 9], [4, 9, 5], [2, 4, 11], [6, 2, 10]]
        + [[8, 6, 7], [9, 8, 1]]
    )
    for _ in range(subdivisions):
        a, b, c = verts[faces[:, 0]], verts[faces[:, 1]], verts[faces[:, 2]]  # (T, 3) each
        ab, bc, ca = (a + b) / 2, (b + c) / 2, (c + a) / 2
        corners = torch.stack([a, ab, ca, ab, b, bc, ca, bc, c, ab, bc, ca], 1).reshape(-1, 3)  # 4 new triangles each
        # Merge the copies of shared vertices: identical rounded coordinates → one index.
        verts, faces = torch.unique((corners * 1e5).round(), dim=0, return_inverse=True)
        verts, faces = verts / 1e5, faces.reshape(-1, 3)
    verts = radius * verts / verts.norm(dim=1, keepdim=True)
    # Spherical texture coordinates per corner; a triangle straddling the u = 0/1 seam gets its small u's lifted by 1.
    p = verts[faces]  # (T, 3, 3)
    u = 0.5 + torch.atan2(p[..., 2], p[..., 0]) / (2 * math.pi)
    v = 0.5 + torch.asin(p[..., 1].clamp(-1, 1)) / math.pi
    seam = (u.max(1, keepdim=True).values - u.min(1, keepdim=True).values) > 0.5
    u = torch.where(seam & (u < 0.5), u + 1, u)
    return verts, faces, torch.stack([4 * u, 2 * v], -1)


def terrain(n: int = 14, size: float = 2.4):
    """Low-poly height field y = h(x, z) on an n × n grid (rolling hills)."""
    s = torch.linspace(-size / 2, size / 2, n + 1)
    xx, zz = torch.meshgrid(s, s, indexing="ij")
    yy = 0.22 * torch.sin(2.3 * xx) * torch.cos(1.9 * zz) + 0.12 * torch.cos(3.1 * xx + 1.7 * zz) - 0.2
    verts = torch.stack([xx, yy, zz], -1).reshape(-1, 3)
    faces, uv = grid_faces(n, n, False, False)
    return verts, faces, uv * 2


MESHES = {"torus": torus, "icosphere": icosphere, "terrain": terrain}


def scene(mesh: str, device: torch.device) -> dict:
    """The mesh plus a 6 × 6 floor made of only two big triangles (where affine texturing goes badly wrong).

    Returns verts (N, 3), faces (T, 3), uv (T, 3, 2), is_mesh_vertex (N,) and is_mesh_face (T,)."""
    mv, mf, muv = MESHES[mesh]()
    fv = torch.tensor([[-3.0, -1.1, -3.0], [-3.0, -1.1, 3.0], [3.0, -1.1, 3.0], [3.0, -1.1, -3.0]])
    ff, fuv = grid_faces(1, 1, False, False)
    # grid_faces' vertex order (i, j) → i·2 + j matches fv as (x index, z index): 0 (−,−), 1 (−,+), 2 (+,−), 3 (+,+).
    fv = fv[[0, 1, 3, 2]]
    verts = torch.cat([mv, fv])
    faces = torch.cat([mf, ff + len(mv)])
    uv = torch.cat([muv, fuv])
    return {
        "verts": verts.to(device),
        "faces": faces.to(device),
        "uv": uv.to(device),
        "is_mesh_vertex": (torch.arange(len(verts)) < len(mv)).to(device),
        "is_mesh_face": (torch.arange(len(faces)) < len(mf)).to(device),
    }


def camera(res: int, device: torch.device) -> dict:
    """Pinhole camera looking at the origin from above and in front; rotation rows = (right, up, forward)."""
    eye = torch.tensor([0.0, 1.5, 4.0])
    forward = torch.tensor([0.0, -0.25, 0.0]) - eye
    forward = forward / forward.norm()
    right = torch.linalg.cross(forward, torch.tensor([0.0, 1.0, 0.0]))
    right = right / right.norm()
    up = torch.linalg.cross(right, forward)
    return {
        "eye": eye.to(device),
        "R": torch.stack([right, up, forward]).to(device),  # world → camera, (3, 3)
        "f": 1.15 * res,  # focal length in pixels (≈ 47° field of view)
        "c": res / 2,  # principal point (centre of the image)
        "res": res,
    }


def pixels(res: int, device: torch.device) -> torch.Tensor:
    """Pixel centres q_{pi} of shape (res·res, 2), row-major from the top row; screen y points up."""
    s = torch.arange(res, device=device, dtype=torch.float32) + 0.5
    yy, xx = torch.meshgrid(res - s, s, indexing="ij")
    return torch.stack([xx, yy], -1).reshape(-1, 2)


def light(device: torch.device) -> torch.Tensor:
    """Direction towards a distant light (unit vector), (3,)."""
    d = torch.tensor([-0.5, 1.0, 0.7], device=device)
    return d / d.norm()


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/graphics/triangle_rasterisation/solve.py
"""The rasteriser: cull, compute barycentrics for (pixels × triangles) in memory-bounded tiles, z-buffer, shade."""

import torch

from problems._common.progress import steps

from .equation import (
    affine_interp,
    barycentric,
    blinn_phong,
    checker,
    inverse_depth,
    perspective_interp,
    project,
    rotate,
    rotation,
    signed_area,
)

NEAR = 0.1
MESH_COLOUR = torch.tensor([0.30, 0.52, 0.95])
FLOOR_COLOUR = torch.tensor([0.80, 0.80, 0.78])
TEXTURE_DARK, TEXTURE_LIGHT = torch.tensor([0.12, 0.25, 0.60]), torch.tensor([0.98, 0.80, 0.35])


def face_normals(verts: torch.Tensor, faces: torch.Tensor) -> torch.Tensor:
    """Area-weighted (un-normalised) face normals (b − a) × (c − a): (T, 3)."""
    a, b, c = verts[faces[:, 0]], verts[faces[:, 1]], verts[faces[:, 2]]
    return torch.linalg.cross(b - a, c - a)


def vertex_normals(verts: torch.Tensor, faces: torch.Tensor) -> torch.Tensor:
    """Average of the normals of the faces around each vertex (needed for smooth shading): (N, 3)."""
    fn = face_normals(verts, faces)
    n = torch.zeros_like(verts).index_add_(0, faces.reshape(-1), fn.repeat_interleave(3, dim=0))
    return n / n.norm(dim=1, keepdim=True).clamp(min=1e-12)


def zbuffer(q: torch.Tensor, tri: torch.Tensor, z: torch.Tensor):
    """Brute force for a set of pixels against a set of triangles: barycentrics (p, T, 3), then the min over T.

    q (p, 2), tri (T, 3, 2), z (T, 3) → index of the nearest covering triangle (p,), its λ (p, 3), its depth (p,)."""
    lam = barycentric(q, tri)  # (p, T, 3)
    inside = (lam >= 0).all(-1)  # (p, T)
    d = torch.where(inside, 1 / inverse_depth(lam, z), float("inf"))  # depth of every covering triangle
    d_min, t = d.min(dim=1)  # the z-buffer: nearest triangle per pixel
    return t, lam[torch.arange(len(t), device=q.device), t], d_min


def rasterise(screen: torch.Tensor, z: torch.Tensor, keep: torch.Tensor, q: torch.Tensor, tile: int = 16):
    """z-buffer every pixel, one screen tile of tile × tile pixels at a time, against only the triangles whose
    bounding box touches that tile (most triangles are small, so this skips almost all pixel-triangle pairs).

    screen (T, 3, 2), z (T, 3), keep (T,) bool, q (P, 2) → triangle id (P,) (−1 = background),
    barycentrics of that triangle (P, 3) and depth (P,)."""
    dev, n_pix = q.device, q.shape[0]
    tri_id = torch.full((n_pix,), -1, dtype=torch.long, device=dev)
    lam_win = torch.zeros(n_pix, 3, device=dev)
    depth = torch.full((n_pix,), float("inf"), device=dev)
    ids = keep.nonzero().squeeze(1)  # (T',) triangles that survive culling
    if len(ids) == 0:
        return tri_id, lam_win, depth
    tri, zt = screen[ids], z[ids]
    lo, hi = tri.min(dim=1).values, tri.max(dim=1).values  # bounding boxes (T', 2)

    cell = torch.floor(q / tile).long()  # tile (column, row) of every pixel, (P, 2)
    n_cols = int(cell[:, 0].max()) + 1
    tile_of_pixel = cell[:, 1] * n_cols + cell[:, 0]  # (P,)
    tiles = torch.unique(tile_of_pixel)
    corner = torch.stack([tiles % n_cols, tiles // n_cols], -1).float() * tile  # lower-left corner of each tile
    # Binning: (tiles × triangles) overlap of bounding boxes, all at once.
    touches = ((lo[None] <= corner[:, None] + tile) & (hi[None] >= corner[:, None])).all(-1)  # (n_tiles, T')
    for k, tid in enumerate(tiles.tolist()):
        cand = touches[k].nonzero().squeeze(1)
        if len(cand) == 0:
            continue
        pix = (tile_of_pixel == tid).nonzero().squeeze(1)
        t, lam, d = zbuffer(q[pix], tri[cand], zt[cand])
        hit = torch.isfinite(d)
        tri_id[pix] = torch.where(hit, ids[cand[t]], -1)
        lam_win[pix], depth[pix] = lam, d
    return tri_id, lam_win, depth


def render(scene: dict, cam: dict, q: torch.Tensor, to_light: torch.Tensor, theta: float, tilt: float) -> dict:
    """Render one frame with the mesh spun by θ. Returns images (res, res, 3) and per-frame statistics."""
    dev, res = q.device, cam["res"]
    R = rotation(theta, tilt, dev)
    verts = torch.where(scene["is_mesh_vertex"][:, None], rotate(R, scene["verts"]), scene["verts"])  # (N, 3)
    faces = scene["faces"]
    normals = vertex_normals(verts, faces)  # (N, 3)

    screen_v, z_v = project(verts, cam)  # (N, 2), (N,)
    screen, z = screen_v[faces], z_v[faces]  # (T, 3, 2), (T, 3)
    front = signed_area(screen) > 1e-9  # back-face culling: clockwise on screen ⇒ facing away
    keep = front & (z.min(1).values > NEAR)
    tri_id, lam, depth = rasterise(screen, z, keep, q)
    hit = tri_id >= 0
    t = tri_id.clamp(min=0)

    # Gather the winning triangle's corner attributes for every pixel: (P, 3, ·).
    zc = z[t]  # (P, 3)
    pos = perspective_interp(lam, zc, verts[faces[t]])  # world position of the visible surface point
    to_eye = cam["eye"] - pos
    to_eye = to_eye / to_eye.norm(dim=1, keepdim=True)
    base = torch.where(scene["is_mesh_face"][t, None], MESH_COLOUR.to(dev), FLOOR_COLOUR.to(dev))  # (P, 3)

    # Flat: one normal per face. Gouraud: light the corners, interpolate the brightness. Phong: interpolate normals.
    fn = face_normals(verts, faces)[t]
    fn = fn / fn.norm(dim=1, keepdim=True)
    centre = verts[faces[t]].mean(1)
    eye_c = (cam["eye"] - centre) / (cam["eye"] - centre).norm(dim=1, keepdim=True)
    flat = blinn_phong(fn, to_light, eye_c)
    corner_pos = verts[faces[t]]  # (P, 3, 3)
    corner_eye = cam["eye"] - corner_pos
    corner_eye = corner_eye / corner_eye.norm(dim=-1, keepdim=True)
    corner_light = blinn_phong(normals[faces[t]], to_light, corner_eye)  # (P, 3)
    gouraud = perspective_interp(lam, zc, corner_light[..., None])[:, 0]
    n_p = perspective_interp(lam, zc, normals[faces[t]])
    n_p = n_p / n_p.norm(dim=1, keepdim=True)
    phong = blinn_phong(n_p, to_light, to_eye)

    # Texture: checker looked up at perspective-correct versus affine texture coordinates.
    uv = scene["uv"][t]  # (P, 3, 2)
    uv_persp, uv_affine = perspective_interp(lam, zc, uv), affine_interp(lam, uv)
    diffuse = 0.45 + 0.55 * (n_p * to_light).sum(-1).clamp(min=0)

    def textured(uv_p):
        c = checker(uv_p)[:, None]
        return (c * TEXTURE_LIGHT.to(dev) + (1 - c) * TEXTURE_DARK.to(dev)) * diffuse[:, None]

    def image(rgb):
        rgb = torch.where(hit[:, None], rgb.clamp(0, 1), torch.ones_like(rgb))  # white background
        return rgb.reshape(res, res, 3)

    near, far = depth[hit].min(), depth[hit].max()
    grey = ((depth - near) / (far - near)).clamp(0, 1)[:, None].expand(-1, 3)
    floor_px = hit & ~scene["is_mesh_face"][t]
    return {
        "depth": image(0.15 + 0.8 * grey),
        "flat": image(flat[:, None] * base),
        "gouraud": image(gouraud[:, None] * base),
        "phong": image(phong[:, None] * base),
        "persp": image(textured(uv_persp)),
        "affine": image(textured(uv_affine)),
        "culled": int((~front).sum()),
        "triangles": len(faces),
        "covered": int(hit.sum()),
        "uv_error": float((uv_persp - uv_affine)[floor_px].norm(dim=1).max()) if floor_px.any() else 0.0,
    }


PANELS = ["depth", "flat", "gouraud", "phong", "persp", "affine"]


def animate_mesh(scene: dict, cam: dict, q, to_light, n_frames: int, tilt: float) -> tuple[torch.Tensor, list]:
    """Render n_frames of one full turn: images (F, 6, res, res, 3) as uint8 and the per-frame statistics."""
    frames, stats = [], []
    for f in steps(n_frames, "frames"):
        out = render(scene, cam, q, to_light, 2 * torch.pi * f / n_frames, tilt)
        frames.append(torch.stack([(out[k] * 255).round().to(torch.uint8) for k in PANELS]).cpu())
        stats.append({k: out[k] for k in ("culled", "triangles", "covered", "uv_error")})
    return torch.stack(frames), stats


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/graphics/triangle_rasterisation/plot.py
"""Six views of the rotating mesh: the z-buffer, three shading models and perspective-correct vs affine texturing."""

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from problems._common.plotting import frame_indices, save_gif

TITLES = [
    "z-buffer depth (dark = near)",
    "flat shading",
    "Gouraud (light corners, blend)",
    "Phong (blend normals, light)",
    "texture, perspective-correct",
    "texture, affine (wrong)",
]


def animate(frames: torch.Tensor, mesh: str, path: Path) -> Path:
    """frames (F, 6, H, W, 3) uint8."""
    imgs = frames.numpy()
    fig, axes = plt.subplots(2, 3, figsize=(8.4, 6.0))
    shown = []
    for ax, title, img in zip(axes.flat, TITLES, imgs[0]):
        shown.append(ax.imshow(img, interpolation="nearest"))
        ax.set_title(title, fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
        for side in ax.spines.values():
            side.set_color("#d1d5db")
    heading = fig.suptitle("", fontsize=11)

    def update(f: int):
        for im, img in zip(shown, imgs[f]):
            im.set_data(img)
        heading.set_text(f"rasterised {mesh}: frame {f + 1}/{len(imgs)}")

    fig.tight_layout(rect=(0, 0, 1, 0.95))
    return save_gif(fig, update, frame_indices(len(imgs), 120), path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'mesh': 'torus',
    'resolution': 128,
    'frames': 48,
    'tilt': 0.5
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.graphics.triangle_rasterisation")]:
    del sys.modules[name]
main = importlib.import_module("problems.graphics.triangle_rasterisation.main")
problem_dir = Path("problems/graphics/triangle_rasterisation")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
